# Goal

В `18n_study_26.6` выяснилось, что с замороженным энкодером жить можно. Даже можно получить некоторое улучшение. Но возникает следующие две проблемы:
- феномен "горбатости". Агент сначала понемного улучшается, а потом деградирует. Скорее это из-за того, что играя долго на поздних уровнях он разучивается хорошо проходить начальные уровни. При этом может пройти дальше, иногда даже 7-ой уровень проходит 
- феномен "флэт". Агент играет ровно и подтягивает нижнюю границу `levels_passed_mean`. В итоге вроде этот показатель растёт, но новые уровни проходить не может.

Вот надо понять - эти феномены из-за замороженного энкодера или это уже предел архитектуры и памяти?

# TARGET_NOTEBOOK_FNAME

In [1]:
TARGET_NOTEBOOK_FNAME = '18n_ppo_tr_frostbite_07.ipynb'

# GRID_SEARCH_SPACE

In [2]:
# @launchit.collect
# GRID_SEARCH_SPACE = dict(
# )

# set_hyperparameters

In [3]:
# @launchit.collect
def set_hyperparameters(HP, optuna_study, optuna_trial):
    import random
    HP.general.random_seed = random.randint(1, 100)
    HP.general.is_torch_deterministic = True
    HP.general.is_torch_compile = True
    HP.general.is_torch_amp = True
    
    HP.env.count = 32 
    HP.env.is_episodic_life = True
    HP.env.actions_count = 6

    HP.vision_head.parent = dict(model='18d_world_model_09:40', weights='18d_world_model_09:40')
    HP.vision_head.is_trainable = False
    
    parent = optuna_trial.suggest_categorical('parent', [
        '18n_ppo_tr_frostbite_07:272', # superstudy_26
        '18n_ppo_tr_frostbite_07:247', # superstudy_25
    ])
    
    HP.encoder.parent = dict(model='18d_world_model_09:40', weights=parent)
    HP.encoder.is_trainable = True

    HP.agent.parent = parent 
    HP.agent.sequence_length = 4
    HP.agent.action_plan_length = 10 
    HP.agent.d_model = 256 
    HP.agent.transformer = dict(layers_count=3, heads_count=4, attention_backend=['EFFICIENT_ATTENTION', 'MATH'])
    HP.agent.is_trainable = True
    
    # Test params
    HP.test.env_rams = None
    HP.test.env_ram_patches = None
    HP.test.break_on_level_passed = False
    
    # Training procedure params (PPO related) 
    HP.ppo.global_steps_count = 3_000_000 # total number of steps 
    HP.ppo.rollout_steps_count = 512 # how many steps to run in a single policy rolllout
    HP.ppo.rollout_env_rams = None
    HP.ppo.rollout_env_ram_patches = [
        ['nine_lives'], 
    ]
    
    HP.ppo.epochs_count = 2 
    HP.ppo.batch_size = 512 
    learn_rate = optuna_trial.suggest_float('learn_rate', 0.00001, 0.0001)
    HP.ppo.learn_rate = f'const({learn_rate})'
    HP.ppo.optimizer = 'AdamW'
    
    HP.ppo.vf_coef = 0.2
    ent_coef = optuna_trial.suggest_float('ent_coef', 0.0023, 0.023)
    HP.ppo.ent_coef = f'const({ent_coef})'
    HP.ppo.consistency_coef = 0.1
    HP.ppo.prediction_coef = 0.1
    
    HP.ppo.gamma = 0.997 # return discount factor gamma
    HP.ppo.gae_lambda = 0.95 # lambda for the general advantage estimation
    HP.ppo.clip_coef = 0.1 # the surrogate clipping coefficient
    HP.ppo.clip_vloss = True
    HP.ppo.max_grad_norm = 0.5 # the maximum norm for the gradient clipping
    HP.ppo.target_kl = None # the target KL divergence threshold
    HP.ppo.norm_adv = True # Toggles advantages normalization
    return HP

# Results


Результаты с т.з. статистики. Джемини говорит, что 26.6 предпочтительнее.

| Metric | Experiment `18n_study_27.6` | Experiment `18n_study_26.6` | Statistical Result / Verdict |
| :--- | :--- | :--- | :--- |
| **Sample Size (\(n\))** | 30 | 30 | — |
| **Mean Score** | 5.0177 | 4.9177 | +0.1000 performance shift favoring `27.6` |
| **Standard Deviation (Noise)** | **0.5364** | **0.2160** | `27.6` is **2.5x noisier** than `26.6` |
| **Shapiro-Wilk (\(p\)-value)** | 0.0040 (Not Normal) | 0.4303 (Normal) | `27.6` completely fails normality (highly skewed) |
| **Welch's \(t\)-test (\(p\)-value)** | — | — | 0.3495 (Not Significant; broken by massive noise asymmetry) |
| **Mann-Whitney U (\(p\)-value)** | — | — | **0.0419** 🟢 (Statistically Significant at \(\alpha = 0.05\)) |
| **Cohen's \(d\) (Effect Size)** | — | — | **0.2446** (Small overall magnitude of effect) |


Здесь также явно наличие двух фракций, как и в `18n_study_26.6`. Но есть отличие: эти фракции теперь выделяются не только по `episode_r`, но и по `levels_passed_mean`. Лидирует фракция, где родителем был `18n_ppo_tr_frostbite_07:272` (флэты). 

Если дать 9 жизней и посмотреть запись видео, то видно различие:
- флэты играют более аккуратно
- зубастики идут напролом

При этом с 9-ью жизнями зубастик скорее доберётся до 8 уровня, тогда как флэт застрянет на 7-ом.

**27.6**
<img src="./img/levels_passed_mean.png">

**26.6**
<img src="./img/levels_passed_mean_26.6.png">

**Выводы**
1) размороженный энкодер даёт небольшой плюс. Хотя это и логично, если учесть что агент добирается до 8-го уровня, где уже поведение может быть другим
2) хотя м.б. нужен новый vision head, обученный с учётом 8-го уровня
3) или нужно и то и другое: и vision head, и энкодер

# System

In [4]:
import os, sys, re, subprocess, json
import IPython 
import concurrent.futures as cf
from collections import namedtuple

import optuna
from optuna.storages import JournalStorage
from optuna.storages.journal import JournalFileBackend
from optuna.trial import TrialState

project_root_path = ! git rev-parse --show-toplevel
project_root_path = project_root_path[0]

sys.path.append(os.path.join(project_root_path, 'lib'))

from logging_utils import *
from math_utils import *
from artifact_registry import *
import launchit
import launch_dispatcher
from autoincrement import Autoincrement

In [5]:
CONFIG = namedtuple('CONFIG', 
                    'project_root_uri, model_group_uri, project_root_path, subproject_name, subproject_path, run_path, ' + 
                    'target_notebook_fname, target_notebook_name, ' + 
                    'optuna_study_notebook_fname, optuna_study_name, optuna_study_serial, optuna_study_fname')(
    project_root_uri=f'com.develorium.{os.path.basename(project_root_path)}',
    model_group_uri=None,
    project_root_path=project_root_path,
    subproject_name=None,
    subproject_path=os.path.abspath('../..'),
    run_path=None,
    target_notebook_fname=os.path.join(os.path.abspath('../..'), TARGET_NOTEBOOK_FNAME),
    target_notebook_name=None,
    optuna_study_notebook_fname=None,
    optuna_study_name=None,
    optuna_study_serial=None,
    optuna_study_fname=None,
)

with open(IPython.get_ipython().kernel.config['IPKernelApp']['connection_file'], 'r') as connection_file:
    optuna_study_notebook_fname = json.load(connection_file).get('jupyter_session')
    optuna_study_name, _ = os.path.splitext(os.path.basename(optuna_study_notebook_fname))
    optuna_study_serial = re.match(r'\w+_([\d\.\w]+)', optuna_study_name).group(1)
    optuna_study_fname = os.path.join(os.path.dirname(optuna_study_notebook_fname), optuna_study_name + '.optuna')
    CONFIG = CONFIG._replace(optuna_study_notebook_fname=optuna_study_notebook_fname)
    CONFIG = CONFIG._replace(optuna_study_name=optuna_study_name)
    CONFIG = CONFIG._replace(optuna_study_serial=optuna_study_serial)
    CONFIG = CONFIG._replace(optuna_study_fname=optuna_study_fname)

target_notebook_name, _ = os.path.splitext(os.path.basename(TARGET_NOTEBOOK_FNAME))
CONFIG = CONFIG._replace(subproject_name=os.path.basename(os.path.dirname(CONFIG.target_notebook_fname)))
CONFIG = CONFIG._replace(model_group_uri=f'{CONFIG.project_root_uri}.{CONFIG.subproject_name}')
CONFIG = CONFIG._replace(target_notebook_name=target_notebook_name)
CONFIG = CONFIG._replace(run_path=os.path.join(project_root_path, 'run', CONFIG.subproject_name))
CONFIG._asdict()

{'project_root_uri': 'com.develorium.neurolab',
 'model_group_uri': 'com.develorium.neurolab.18_rl',
 'project_root_path': '/home/misha/dev/mine/neurolab',
 'subproject_name': '18_rl',
 'subproject_path': '/home/misha/dev/mine/neurolab/18_rl',
 'run_path': '/home/misha/dev/mine/neurolab/run/18_rl',
 'target_notebook_fname': '/home/misha/dev/mine/neurolab/18_rl/18n_ppo_tr_frostbite_07.ipynb',
 'target_notebook_name': '18n_ppo_tr_frostbite_07',
 'optuna_study_notebook_fname': '/home/misha/dev/mine/neurolab/18_rl/optuna/18n_study_27.6/18n_study_27.6.ipynb',
 'optuna_study_name': '18n_study_27.6',
 'optuna_study_serial': '27.6',
 'optuna_study_fname': '/home/misha/dev/mine/neurolab/18_rl/optuna/18n_study_27.6/18n_study_27.6.optuna'}

In [6]:
LOG = Logging.get()
LOG.enable('syslog', False)
LOG.enable('stdout', False)
LOG.enable('verbose_stdout', True)

In [7]:
ARTIFACT_REGISTRY = ArtifactRegistry(maven_group_id=CONFIG.model_group_uri)

In [8]:
def create_optuna_launch():
    model_version = int(Autoincrement.get(f'{CONFIG.model_group_uri}.{CONFIG.target_notebook_name}'))
    assert model_version > 0, model_version
    ARTIFACT_REGISTRY.register_component(CONFIG.target_notebook_name, model_version)
    LOG(f'Model instance registered, version={model_version}')
    
    # Prep docker launch
    expandvars = dict(
        PROJECT_ROOT_PATH='/neurolab',
        BUILD_PROJECT_ROOT_PATH=CONFIG.project_root_path,
        MODEL_GROUP_URI=CONFIG.model_group_uri,
        MODEL_NAME=CONFIG.target_notebook_name,
        MODEL_VERSION=model_version,
        LAUNCH_GOAL='TRAIN',
        OPTUNA_STUDY_FNAME=CONFIG.optuna_study_fname,
        OPTUNA_STUDY_NAME=CONFIG.optuna_study_name,
        OPTUNA_DECISIVE_METRIC='test/levels_passed_mean',
    )
    launch_fname = launchit.launchit(
        CONFIG.target_notebook_fname, 
        launch_serial=int(model_version),
        expandvars=expandvars, 
        make_py_file=False, 
        dir_name=CONFIG.run_path,
        collect_inds=['temp_config', 'optuna', 'initrd', 'build_docker_launch', 'optuna_run_docker_launch'],
        disable_inds=[],
    )
    return f'{CONFIG.target_notebook_name}:{model_version}', launch_fname

In [9]:
# Executed in a separate thread with GIL locked
def run_optuna_launch(launch_fname):
    # Run launch notebook locally, the latter will:
    # 1) sample values of hyperparameters from optuna study
    # 2) pack everything to docker launch (self-contained thing)
    # 3) run "docker_launch_run" cell which in turn will dispatch launch to cloud via launch_dispatcher
    # 4) collect result of a docker launch from cloud and update optuna study
    LOG(f'Launching "{launch_fname}"')
    
    subprocess.run(
        ['papermill', launch_fname, launch_fname, '--no-progress-bar'],
        capture_output=False,
        stdout=subprocess.DEVNULL,
        stderr=subprocess.DEVNULL,
        check=True,
    )

    if os.path.exists(launch_fname + '.out'):
        with open(launch_fname + '.out', 'rt') as f:
            LOG(f.read())
    else:
        LOG(f'"{launch_fname}" completed with no output, probably failed')

## Unleash!

In [10]:
optuna_study = optuna.create_study(
    study_name=CONFIG.optuna_study_name,
    directions=['maximize'],
    storage=JournalStorage(JournalFileBackend(file_path=CONFIG.optuna_study_fname)),
    load_if_exists=True,
)
optuna_study.set_user_attr('STUDY_SERIAL', CONFIG.optuna_study_serial)
launches_count = 30
completed_launches_count = 0

with LOG.auto_log_level(logging.INFO):
    with cf.ThreadPoolExecutor(max_workers=32) as executor:
        futures = {}
        idle_runners_af = RecursiveMovingAverageFilter(max_n=6)
        is_first_time = True
        
        while launches_count is None or completed_launches_count < launches_count:
            runners_info = launch_dispatcher.RunnersInfo.get()
            idle_runners_af(runners_info['idle'])

            if is_first_time or (idle_runners_af.n >= idle_runners_af.max_n and idle_runners_af.v >= 1):
                if launches_count is None or (completed_launches_count + len(futures) < launches_count):
                    launch_name, launch_fname = create_optuna_launch()
                    futures.update({executor.submit(run_optuna_launch, launch_fname): launch_name})
                    LOG(f'{idle_runners_af.v:.1f} idle runners exist, submitted launch "{launch_name}"; running launches={len(futures)}')
                    idle_runners_af.reset()
                    
                is_first_time = False

            try:
                while futures:
                    completed_futures, _ = cf.wait(futures, timeout=0.1, return_when=cf.FIRST_COMPLETED)

                    if not completed_futures:
                        break
                        
                    for completed_future in completed_futures:
                        launch_name = futures[completed_future]
                        del futures[completed_future]

                        exc = completed_future.exception()
                        
                        if exc is not None:
                            LOG(f'Launch "{launch_name}" failed: {exc}')
                        else:
                            LOG(f'Launch "{launch_name}" completed')
    
                    if completed_futures:
                        completed_launches_count += len(completed_futures)
                        LOG(f'{completed_launches_count} (+{len(completed_futures)}) launches completed; running launches={len(futures)}')
            except TimeoutError as e:
                pass

            time.sleep(5)

[I 2026-09-28 18:59:07,199] A new study created in Journal with name: 18n_study_27.6


2026.09.28-18:59:07.482598     0.259 >> Model instance registered, version=321
2026.09.28-18:59:07.498654     0.006 >> Launching "/home/misha/dev/mine/neurolab/run/18_rl/18n_ppo_tr_frostbite_07-launch321.ipynb"
2026.09.28-18:59:07.499208     0.007 >> 7.0 idle runners exist, submitted launch "18n_ppo_tr_frostbite_07:321"; running launches=1
2026.09.28-18:59:38.482951     0.230 >> Model instance registered, version=322
2026.09.28-18:59:38.491713     0.004 >> Launching "/home/misha/dev/mine/neurolab/run/18_rl/18n_ppo_tr_frostbite_07-launch322.ipynb"
2026.09.28-18:59:38.492028     0.004 >> 5.2 idle runners exist, submitted launch "18n_ppo_tr_frostbite_07:322"; running launches=2
2026.09.28-19:00:09.601830     0.238 >> Model instance registered, version=323
2026.09.28-19:00:09.614511     0.004 >> Launching "/home/misha/dev/mine/neurolab/run/18_rl/18n_ppo_tr_frostbite_07-launch323.ipynb"
2026.09.28-19:00:09.614809     0.004 >> 3.3 idle runners exist, submitted launch "18n_ppo_tr_frostbite_07

In [11]:
study = optuna.create_study(
    study_name=optuna_study_name,
    storage=JournalStorage(JournalFileBackend(file_path=optuna_study_fname)),
    load_if_exists=True, 
)

pruned_trials = study.get_trials(deepcopy=False, states=[TrialState.PRUNED])
complete_trials = study.get_trials(deepcopy=False, states=[TrialState.COMPLETE])

LOG('Study statistics: ')
LOG(f'\tNumber of finished trials: {len(study.trials)}')
LOG(f'\tNumber of pruned trials: {len(pruned_trials)}')
LOG(f'\tNumber of complete trials: {len(complete_trials)}')

if len(study.directions) == 1:
    LOG('Best trial:')
    trial = study.best_trial
    
    LOG(f'\tValue: {trial.value}')
    LOG(f'\tModel version: {trial.user_attrs.get('MODEL_VERSION', 'n/a')}')
    
    LOG('\tParams: ')
    
    for key, value in trial.params.items():
        LOG(f'\t\t{key}: {value}')
else:
    LOG(f"Number of trials on the Pareto front: {len(study.best_trials)}")

    for i in range(3):
        LOG(f"Trial with lowest loss_{i}:")
        trial = min(study.best_trials, key=lambda t: t.values[i])
        LOG(f"\tnumber: {trial.number}")
        LOG(f"\tmver: {trial.user_attrs['MODEL_VERSION']}")
        LOG(f"\tparams: {trial.params}")
        LOG(f"\tvalues: {trial.values}")

[I 2026-09-28 21:36:44,732] Using an existing study with name '18n_study_27.6' instead of creating a new one.


2026.09.28-21:36:44.734283     5.032 >> Study statistics: 
2026.09.28-21:36:44.738111     0.004 >> 	Number of finished trials: 30
2026.09.28-21:36:44.739255     0.001 >> 	Number of pruned trials: 0
2026.09.28-21:36:44.740368     0.001 >> 	Number of complete trials: 30
2026.09.28-21:36:44.741518     0.001 >> Best trial:
2026.09.28-21:36:44.742542     0.001 >> 	Value: 5.65625
2026.09.28-21:36:44.743224     0.001 >> 	Model version: 321
2026.09.28-21:36:44.744026     0.001 >> 	Params: 
2026.09.28-21:36:44.744587     0.001 >> 		parent: 18n_ppo_tr_frostbite_07:272
2026.09.28-21:36:44.745223     0.001 >> 		learn_rate: 1.821859428788061e-05
2026.09.28-21:36:44.745617     0.000 >> 		ent_coef: 0.0068316668973403515


In [12]:
top_trials = sorted(filter(lambda t: t.values, study.trials), key=lambda t: -t.values[0])[:3]
LOG(f'Top trials:')

for i, trial in enumerate(top_trials):
    LOG(f'{i+1}) {CONFIG.target_notebook_name + ':' + trial.user_attrs['MODEL_VERSION']}, value={trial.values[0]}')

2026.09.28-21:36:44.753961     0.008 >> Top trials:
2026.09.28-21:36:44.754548     0.001 >> 1) 18n_ppo_tr_frostbite_07:321, value=5.65625
2026.09.28-21:36:44.754991     0.000 >> 2) 18n_ppo_tr_frostbite_07:330, value=5.65625
2026.09.28-21:36:44.755258     0.000 >> 3) 18n_ppo_tr_frostbite_07:325, value=5.625


In [18]:
values = []

for trial in study.trials:
    if trial.values is not None:
        values.append(trial.values[0])

values = np.array(values)
values_27_6 = values

In [17]:
study_26_6 = optuna.create_study(
    study_name='18n_study_26.6',
    storage=JournalStorage(JournalFileBackend(file_path=os.path.join(os.path.abspath('..'), '18n_study_26.6', '18n_study_26.6.optuna'))),
    load_if_exists=True, 
)

values_26_6 = []

for trial in study_26_6.trials:
    if trial.values is not None:
        values_26_6.append(trial.values[0])

values_26_6 = np.array(values_26_6)

[I 2026-09-28 21:40:42,844] Using an existing study with name '18n_study_26.6' instead of creating a new one.


In [25]:
import scipy.stats as stats
import numpy as np
import pandas as pd
from collections import defaultdict

columns = defaultdict(list)
columns['name'].extend(['27.6 (current)', '26.6 (prev)', '-'])

# Basic metrics
mean_27_6 = np.mean(values_27_6)
mean_26_6 = np.mean(values_26_6)
columns['mean'].extend([mean_27_6, mean_26_6, None])

std_27_6 = np.std(values_27_6, ddof=1)
std_26_6 = np.std(values_26_6, ddof=1)
columns['std'].extend([std_27_6, std_26_6, None])

# Normality Check (Shapiro-Wilk)
shapiro_27_6 = stats.shapiro(values_27_6)
shapiro_26_6 = stats.shapiro(values_26_6)
columns['shapiro'].extend([shapiro_27_6.pvalue, shapiro_26_6.pvalue, None])

# Welch's t-test
t_stat, p_welch = stats.ttest_ind(values_27_6, values_26_6, equal_var=False)
columns['welch'].extend([None, None, p_welch])

# Mann-Whitney U test
u_stat, p_mw = stats.mannwhitneyu(values_27_6, values_26_6, alternative='two-sided')
columns['mann-whitney'].extend([None, None, p_mw])

# Cohen's d
pooled_std = np.sqrt(((len(values_27_6)-1)*std_27_6**2 + (len(values_26_6)-1)*std_26_6**2) / (len(values_27_6) + len(values_26_6) - 2))
cohens_d = (mean_27_6 - mean_26_6) / pooled_std
columns['cohen d'].extend([None, None, cohens_d])

pd.DataFrame(columns)

,name,mean,std,shapiro,welch,mann-whitney,cohen d
0,27.6 (current),5.017708,0.536371,0.004015,NaN,NaN,NaN
1,26.6 (prev),4.917708,0.215959,0.430257,NaN,NaN,NaN
2,-,NaN,NaN,NaN,0.349464,0.041853,0.244583


In [30]:
','.join(map(str, values_27_6.tolist()))

'5.65625,5.28125,3.8125,5.15625,5.625,5.5,5.125,4.25,4.15625,5.65625,5.09375,4.1875,4.125,5.375,5.53125,5.25,4.34375,4.5,4.40625,4.96875,5.5,5.59375,5.28125,5.375,4.96875,5.28125,5.15625,5.28125,4.84375,5.25'

In [29]:
','.join(map(str, values_26_6.tolist()))

'4.46875,4.71875,4.75,4.75,4.84375,5.09375,5.0625,4.625,5.21875,4.84375,4.78125,4.53125,5.25,4.71875,5.0,5.125,4.96875,4.84375,5.21875,4.75,5.0,4.71875,5.28125,5.0,4.9375,5.0,5.15625,4.75,5.03125,5.09375'